# Preparación de modelado — tabla, splits y por qué son distintos a los de Fase 4

## Qué responde este notebook

Fase 4 dejó ~32 features seleccionadas con evidencia (`11_seleccion_features.ipynb`). Fase 5
empieza aquí: una sola tabla de modelado reusable (no repetida a mano en cada notebook siguiente,
mismo motivo que ya llevó a `agregar_promedios_jugador()` en Fase 2), y los splits temporales que
va a usar el resto de la fase — que **no son los mismos** que usó Fase 4, por una razón concreta
que se explica abajo, no por descuido.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import pandas as pd

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
print("filas:", wr.shape[0], "columnas:", wr.shape[1])

filas: 23610 columnas: 268


## 1. El feature set — el mismo de Fase 4, en dos versiones

`features.columnas_por_modelo("arbol")` (34 columnas: la unión de las 15 más importantes de cada
target en `11_seleccion_features.ipynb`, más `edad`/`anios_experiencia`, confirmadas con evidencia
individual aunque justo fuera del top 15) y `columnas_por_modelo("ridge")` (30 columnas: se quitan
las versiones de `target_share`/`air_yards_share` — colineales con `wopr`, ver Fase 3 — y se
agregan `anios_experiencia_sq`/`_bucket`, que solo un modelo lineal necesita para capturar la curva
que un árbol ya ve directo en el dato crudo).

In [2]:
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
print(f"arbol: {len(features_arbol)} columnas")
print(f"ridge: {len(features_ridge)} columnas ({len(set(features_arbol) - set(features_ridge))} se quitan, "
      f"{len(set(features_ridge) - set(features_arbol))} se agregan)")
faltantes = [c for c in set(features_arbol) | set(features_ridge) if c not in wr.columns]
print("columnas declaradas pero ausentes en la tabla (debe ser vacío):", faltantes)

arbol: 34 columnas
ridge: 30 columnas (6 se quitan, 2 se agregan)
columnas declaradas pero ausentes en la tabla (debe ser vacío): []


## 2. Por qué el split de Fase 5 es distinto al de Fase 4

Fase 4 usó train ≤2023 / validación 2024-2025, sin conjunto de prueba — correcto para *screening*
de features con un modelo fijo, porque ahí no se elegía entre familias de modelo ni se buscaban
hiperparámetros. Fase 5 sí hace las dos cosas — usar el mismo conjunto de validación para "elegir
qué features sirven" (Fase 4) y para "elegir qué modelo gana" (Fase 5) sería mirar el mismo dato
dos veces para dos decisiones distintas, una forma de fuga de decisión aunque no sea fuga de datos
en el sentido estricto. Por eso Fase 5 usa 3 particiones, no 2:

- **Train**: 2016-2021 (6 temporadas).
- **Validación**: 2022-2023 — aquí se elige el modelo ganador por target.
- **Prueba**: 2024-2025 — se mira una sola vez, al final, para reportar el resultado honesto.

In [3]:
TRAIN_SEASONS = range(2016, 2022)
VAL_SEASONS = [2022, 2023]
TEST_SEASONS = [2024, 2025]

train_mask, val_mask, test_mask = modeling.dividir_temporal(wr, TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS)
print(f"train: {train_mask.sum()} filas (2016-2021)")
print(f"val:   {val_mask.sum()} filas (2022-2023)")
print(f"test:  {test_mask.sum()} filas (2024-2025)")
print(f"filas fuera de las 3 particiones: {(~(train_mask | val_mask | test_mask)).sum()} "
      f"(debe ser 0 -- esta tabla solo trae 2016-2025; 2026 se carga aparte en el notebook 18, "
      f"con el esquema de depth chart 'actual', no el histórico)")

train: 13795 filas (2016-2021)
val:   4863 filas (2022-2023)
test:  4952 filas (2024-2025)
filas fuera de las 3 particiones: 0 (debe ser 0 -- esta tabla solo trae 2016-2025; 2026 se carga aparte en el notebook 18, con el esquema de depth chart 'actual', no el histórico)


## 3. Nulos por familia de modelo — ejemplo real antes de decidir cómo se manejan

Antes de imputar nada, vale la pena ver un caso real de por qué hay `NaN` en el feature set: la
primera fila de un jugador en la tabla no tiene semana anterior de la que promediar.

In [4]:
primera_fila = wr.sort_values(["player_id", "season", "week"]).groupby("player_id").head(1)
ejemplo = primera_fila[primera_fila["player_display_name"].notna()].iloc[0]
cols_ejemplo = ["player_display_name", "season", "week", "anios_experiencia", "targets_season_avg",
                "targets_last3_avg", "targets_last5_avg", "targets_career_avg", "depth_team"]
print(ejemplo[cols_ejemplo])
print(f"\n% de nulos en cada feature de árbol, sobre toda la tabla:")
print(wr[features_arbol].isna().mean().sort_values(ascending=False).round(3).head(10))

player_display_name    Steve Smith
season                        2016
week                             1
anios_experiencia               15
targets_season_avg             NaN
targets_last3_avg              NaN
targets_last5_avg              NaN
targets_career_avg             NaN
depth_team                     1.0
Name: 0, dtype: object

% de nulos en cada feature de árbol, sobre toda la tabla:
depth_team                        0.191
fantasy_points_ppr_season_avg     0.098
receptions_season_avg             0.098
air_yards_share_season_avg        0.098
receiving_tds_season_avg          0.098
targets_season_avg                0.098
receiving_yards_season_avg        0.098
receiving_air_yards_season_avg    0.098
receiving_epa_last3_avg           0.085
catch_rate_career_avg             0.052
dtype: float64


El ejemplo real no es un rookie — `anios_experiencia = 15` en esa fila: es Steve Smith Sr., ya
veterano en 2016, la primera temporada que cubren nuestros datos. Su `NaN` no significa "no tiene
carrera anterior" — significa que **no tenemos su historia real de antes de 2016** para promediar,
un límite de cobertura de los datos (truncamiento por la izquierda), no una fuga ni un error. Un
rookie de verdad tendría el mismo `NaN` por una razón distinta (sí es su primera semana real) —
ambos casos se tratan igual porque el resultado correcto es el mismo: no hay nada real que
promediar todavía. `depth_team` es el que más nulos tiene en general, por su cobertura incompleta
(0% en 2025+, ~90% en años viejos, documentado desde Fase 4).

**Manejo por familia de modelo** (aplicado dentro de `modeling.entrenar_y_evaluar`, no aquí):
- **XGBoost / HistGradientBoosting**: manejan `NaN` de forma nativa — no se imputa nada, el árbol
  aprende una dirección para los valores faltantes en cada split.
- **RandomForest**: `sklearn` no acepta `NaN` — se imputa con la mediana de cada columna
  (`SimpleImputer`), calculada solo sobre el conjunto de entrenamiento.
- **Ridge**: mismo `SimpleImputer` de mediana, más escalado (`StandardScaler`) — a diferencia del
  pipeline heredado, que imputaba pero nunca escalaba (ver notebook `13_comparacion_modelos.ipynb`
  para el porqué de esta corrección).
- **Baseline**: no necesita imputación de features — solo usa el propio promedio del jugador, con
  su propio fallback en cascada (`season_avg` → `career_avg` → media de entrenamiento).

## Cierre

Tabla de modelado (`features.construir_tabla_modelado`), feature sets por familia
(`features.columnas_por_modelo`), y los 3 splits temporales de Fase 5 — listos para que
`13_comparacion_modelos.ipynb` los consuma sin repetir este bloque. Ningún resultado de modelo
todavía: eso empieza en el siguiente notebook.

Siguiente: [`13_comparacion_modelos.ipynb`](13_comparacion_modelos.ipynb).